# 异常控制

## 异常传播
- 异常被抛出后会沿着调用堆栈一级级传播
1. 异常抛出后，当前帧中断，沿调用栈向上传播（栈展开），逐层查找能匹配的 `except`
2. 找到匹配的 `except`：
    - 控制流跳到该块，异常不再向上传播
    - 传播途中遇到 `finally`、`with` 仍会执行清理
3. 没找到：继续到顶层主线程脚本中，解释器打印 traceback 到 `stderr`，程序退出
   - 交互式解释器：打印 traceback，但回到提示符
   - 非主线程：通常只终止该线程，不一定终止整个程序

## 异常捕获和处理
- 使用try、except、else、finally协同捕获和处理异常
    1. try：运行可能失败的代码
    2. except：失败时处理
    3. else：成功时处理
    4. finally：最后一定要做的收尾
### try：用于包裹可能会出错的语句
- 当其中某一行出现报错时，立即终止，剩下的语句不执行
- 如果try中代码全部执行成功，进入else代码块
- 如果try中代码出现报错，且
    - except成功捕获，进入except代码块
    - except未能捕获，相当于异常逃逸，未能控制
- 最终执行finally代码块
### except：匹配异常类型
- 在except后面接预计可能出现的异常类型，尝试捕获try中出现的异常
- 可以使用元组，一次匹配多个异常类型
- 使用`as exc`取得异常对象
    - ValueError -> 要匹配的异常类型
    - exc -> 实际捕获到的异常对象

- `except (ValueError, TypeError) as exc:`
    - `print(f"转换失败：{exc}")`

### else:只有成功才允许执行
- 一段有可能失败的业务逻辑可以分成try和else
    - try放可能失败的操作
    - else放后置的，没有失败的操作（常常是对try的成果继续后处理）
- 还能避免误用旧值，如果else代码块的内容放在外部，则会在失败的轮次使用之前成功时残留的变量旧值（想起了循环变量的坑）


### finally
- 在正常流程中总是会执行，常用于记录处理后的结束状态

### 异常捕获的边界
- 可以接管异常控制流，输出失败原因，避免程序中断
- 不能等同于修复异常

In [ ]:
successes = []
failures = []
rows = [1,'12',5,'hello',[1,2,3]]
for row in rows:
    try:
        result = float(row)
    except (ValueError, TypeError) as exc:
        failures.append({
            "row": row,
            "error": str(exc),
        })
    else:
        successes.append(result)

print(successes)
print(failures)
# {'row': 'hello', 'error': "could not convert string to float: 'hello'"},
# {'row': [1, 2, 3], 'error': "float() argument must be a string or a real number, not 'list'"}

## 主动抛出、重抛与异常链

### 主动抛出异常
- 面对业务逻辑上的异常，可以使用`raise`主动抛出
- 如果不做处理，后续正常代码也会停止执行

In [ ]:
amount = -20

if amount < 0:
    raise ValueError("amount 不能为负数")

### 重抛
- 如果捕获后不能接管，只能够记录，使用无参数`raise`重抛
- 重抛后：
1. 当前 except 块立即结束，异常继续沿调用栈向上传播
2. 当前 except 不会再次捕获它；外层 try/except 可以捕获
3. 传播途中经过的 finally、with 仍会执行清理
4. 若一直无人捕获，最终到顶层由解释器打印 traceback 并终止主程序
    - 重抛的是当前异常对象，保留原始 traceback
- 无参数 `raise` 只能用在 except 块中，否则报 `RuntimeError: No active exception to re-raise`

### 自定义异常
- 结合异常类定义和raise，转换异常到上层业务语义
- 使用from exc，在表达上层语义的同时保留底层技术原因（链式异常）

In [ ]:
class RecordError(Exception):
    # 让 RecordError 成为 Exception 的子类，从而间接成为 BaseException 的子类
    pass


def parse_amount(row):
    try:
        amount = float(row["amount"])
    except (KeyError, TypeError, ValueError) as exc:
        raise RecordError("amount 字段无法转换") from exc
        # 创建一个 RecordError 异常实例，消息是 "amount 字段无法转换"
        # 把 exc 设为它的 __cause__
        # 然后抛出
    if amount < 0:
        raise RecordError("amount 不能为负数")

    return amount

row1 = {"amount":100}
row2 = {"amount":"hello"}
row3 = {"amount":-100}
parse_amount(row2)

### 断言
- 语法：`assert condition, "失败说明"`
    - 条件为真时，无事发生
    - 条件为假时，抛出 AssertionError
- `-O`运行时，断言语句不会编译执行，等于没写
- 因此不适合用于校验外部数据或者必要发生的校验逻辑

| 场景 | 应使用 |
|---|---|
| 外部输入校验 | `if` 加 `raise` |
| 业务规则校验 | `if` 加 `raise` |
| 开发期内部不变量 | `assert` |

In [ ]:
def sort_pair(a, b):
    if a < b: # 故意写错
        a, b = b, a
    assert a <= b, "交换后必须有序"
    return a, b

# 如果抛出断言异常，说明逻辑写错了
sort_pair(1,2) # 调用函数，运行到assert时抛出断言异常